# Data Preparation — S2W8A2 Stage 2

WILDA Team 4 · Customer Churn Analysis.

Produces the Data Preparation deliverables:

1. A preprocessed dataset with missing values handled and categorical variables encoded
2. Training and testing sets for model validation
3. Scaling applied to normalise the data

**Before running:** place the team's raw dataset at `data/Dataset_ATS_v2.csv`.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.2

ROOT = Path.cwd().parent if Path.cwd().name == "Data_Preparation" else Path.cwd()
RAW = ROOT / "data" / "Dataset_ATS_v2.csv"
OUT = ROOT / "Data_Preparation"

pd.set_option("display.max_columns", 50)

## 1. Load the raw data

In [ ]:
if not RAW.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at {RAW}\n"
        "Place the team's dataset there (or edit RAW above) and re-run."
    )

df = pd.read_csv(RAW)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# Set these to match the real dataset before running the cells below.
TARGET = "Churn"          # the churn column
ID_COLS = ["customerID"]  # identifier columns to exclude from modelling

df.info()

## 2. Handle missing data

Inspect what is missing first, then choose a treatment per column and record the
reason — the video walkthrough needs you to justify these choices.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing values by column:")
print(missing if len(missing) else "  none")
print(f"\nRows with at least one missing value: {df.isna().any(axis=1).sum():,}")

In [ ]:
# Some churn datasets store TotalCharges as text with blanks for new customers.
# Coerce to numeric so those blanks surface as NaN rather than hiding as strings.
# `exclude` rather than `include="object"`: pandas 3 reports text columns as
# dtype 'str', so an include-list written against pandas 2 silently matches none.
for col in df.select_dtypes(exclude=["number", "bool"]).columns:
    if col in ID_COLS or col == TARGET:
        continue
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.notna().mean() > 0.9:
        print(f"{col}: treating as numeric ({converted.isna().sum()} blanks -> NaN)")
        df[col] = converted

In [ ]:
# Numeric gaps: median, which is robust to the skew typical of billing columns.
# Categorical gaps: mode, the most common category.
before = int(df.isna().sum().sum())

for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            fill = df[col].median()
            how = f"median {fill:,.2f}"
        else:
            fill = df[col].mode().iloc[0]
            how = f"mode {fill!r}"
        df[col] = df[col].fillna(fill)
        print(f"{col}: filled with {how}")

print(f"\nMissing values: {before} -> {int(df.isna().sum().sum())}")

## 3. Encode categorical variables

One-hot encoding with `drop_first=True` avoids the dummy-variable trap (perfect
collinearity between the generated columns).

In [ ]:
y = df[TARGET]
if not pd.api.types.is_numeric_dtype(y):
    y = (y.astype(str).str.strip().str.lower().isin(["yes", "true", "1"])).astype(int)
print(f"Target '{TARGET}': {y.mean():.1%} churn rate, {len(y):,} rows")

X = df.drop(columns=ID_COLS + [TARGET], errors="ignore")
cat_cols = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()
print(f"Categorical columns to encode ({len(cat_cols)}): {cat_cols}")

In [ ]:
X = pd.get_dummies(X, columns=cat_cols, drop_first=True).astype(float)
print(f"After encoding: {X.shape[1]} features")
X.head()

In [ ]:
# Deliverable 1 — the preprocessed dataset.
preprocessed = X.copy()
preprocessed[TARGET] = y.values
OUT.mkdir(parents=True, exist_ok=True)
preprocessed.to_csv(OUT / "preprocessed_dataset.csv", index=False)
print(f"Saved preprocessed_dataset.csv  {preprocessed.shape}")

## 4. Train / test split

Stratified on the target so both sets keep the same churn rate — important when
churn is the minority class.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {len(X_train):,} rows ({1 - TEST_SIZE:.0%})  churn {y_train.mean():.1%}")
print(f"Test:  {len(X_test):,} rows ({TEST_SIZE:.0%})  churn {y_test.mean():.1%}")

## 5. Scaling

`StandardScaler` centres each feature at zero with unit variance.

**The scaler is fitted on the training set only**, then applied to the test set.
Fitting on the full dataset would leak information about the test set into
training and inflate the results.

In [ ]:
scaler = StandardScaler().fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print("Training set after scaling (should be ~0 mean, ~1 std):")
X_train_scaled.describe().loc[["mean", "std"]].T.head(8)

In [ ]:
# Deliverable 2 — the training and testing sets.
X_train_scaled.assign(**{TARGET: y_train.values}).to_csv(OUT / "train_set.csv", index=False)
X_test_scaled.assign(**{TARGET: y_test.values}).to_csv(OUT / "test_set.csv", index=False)

# Deliverable 3 — the fitted scaler, so the clustering notebook reuses it exactly.
joblib.dump(scaler, OUT / "scaler.pkl")
joblib.dump(list(X.columns), OUT / "feature_names.pkl")

print("Saved train_set.csv, test_set.csv, scaler.pkl, feature_names.pkl")

## 6. Summary for `scaling_techniques.pdf`

The cell below prints the numbers the required PDF has to document. Paste them
into the document along with your reasoning for choosing StandardScaler over
MinMaxScaler or RobustScaler.

In [ ]:
summary = f"""
DATA PREPARATION SUMMARY - WILDA Team 4, Stage 2
================================================
Raw dataset:          {df.shape[0]:,} rows x {df.shape[1]} columns
Missing values:       {before} found, 0 remaining
Encoding:             one-hot, drop_first=True
                      {len(cat_cols)} categorical columns -> {X.shape[1]} features
Split:                {1 - TEST_SIZE:.0%} train / {TEST_SIZE:.0%} test, stratified on {TARGET}
                      {len(X_train):,} train rows, {len(X_test):,} test rows
Scaling:              StandardScaler (zero mean, unit variance)
                      fitted on the TRAINING SET ONLY, then applied to test
Random state:         {RANDOM_STATE} (results are reproducible)
"""
print(summary)
(OUT / "preparation_summary.txt").write_text(summary, encoding="utf-8")